# CNC Guard — AI4I offline azterketa eta entrenamendua

Une bereko egoeraren sailkapena; ez 24 orduko aurreikuspena. Jatorrizko TEMPLATE-a mantendu da.
Entrenamendua: `python -m scripts.train_model`, biltegiaren errotik. Notebook honek gordetako test emaitzak irakurtzen ditu; ez du eredua berriro doitzen.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'data/ai4i2020_train.csv').exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / 'data/ai4i2020_train.csv')
print(df.shape)
print(df.dtypes.to_string())

(8000, 13)
machine_id             str
machine_type           str
air_temp_K         float64
process_temp_K     float64
rpm                  int64
torque_Nm          float64
tool_wear_min        int64
machine_failure      int64
failure_TWF          int64
failure_HDF          int64
failure_PWF          int64
failure_OSF          int64
failure_RNF          int64


## Datuen azterketa

Falta diren balioak, bikoiztuak, klase-inoreka eta sentsoreen barrutiak. Ez erabili azpimotak iragarle gisa.

In [2]:
print('Missing:', df.isna().sum().to_dict())
print('Duplicates:', df.duplicated().sum())
print('Target:', df.machine_failure.value_counts().to_dict())
features = ['air_temp_K','process_temp_K','rpm','torque_Nm','tool_wear_min']
print(df[features].describe().to_string())
print('Sensor correlations:')
print(df[features].corr().round(3).to_string())

Missing: {'machine_id': 0, 'machine_type': 0, 'air_temp_K': 0, 'process_temp_K': 0, 'rpm': 0, 'torque_Nm': 0, 'tool_wear_min': 0, 'machine_failure': 0, 'failure_TWF': 0, 'failure_HDF': 0, 'failure_PWF': 0, 'failure_OSF': 0, 'failure_RNF': 0}
Duplicates: 0
Target: {0: 7729, 1: 271}
        air_temp_K  process_temp_K          rpm    torque_Nm  tool_wear_min
count  8000.000000     8000.000000  8000.000000  8000.000000    8000.000000
mean    300.005450      310.006062  1539.356875    40.003363     107.685000
std       1.996844        1.479432   180.982943    10.019546      63.612002
min     295.300000      305.700000  1168.000000     3.800000       0.000000
25%     298.300000      308.800000  1422.000000    33.200000      53.000000
50%     300.100000      310.100000  1503.000000    40.100000     107.000000
75%     301.500000      311.100000  1613.000000    46.800000     163.000000
max     304.500000      313.800000  2886.000000    76.600000     253.000000
Sensor correlations:
             

## Aurreprozesatzea eta ebaluazioa

70/15/15 ausazko zatiketa estratifikatua, seed=42. Medianak eta eskalatzea entrenamenduan soilik.
RF eta erregresio logistikoan class_weight=balanced, gehiengoaren aldeko joera arintzeko.
RF atalasea validazioan F1 maximizatuz; test emaitzak ez dira hautatzeko erabiltzen.
Xehetasunak: `scripts/train_model.py` eta `docs/data/ENTRENAMENDUA.md`.

In [3]:
report = json.loads((ROOT / 'models/offline-ai4i-v1/report.json').read_text())
print('Versions:', report['versions'])
for name, part in report['split'].items():
    print(name, part['rows'], 'rows;', part['positives'], 'failures')
for name, result in report['models'].items():
    print(name, json.dumps(result['test'], indent=2))

Versions: {'python': '3.14.8', 'scikit-learn': '1.9.1', 'pandas': '3.0.0', 'numpy': '2.4.2', 'joblib': '1.6.0'}
train 5600 rows; 190 failures
validation 1200 rows; 41 failures
test 1200 rows; 40 failures
always_zero {
  "threshold": 0.5,
  "precision": 0.0,
  "recall": 0.0,
  "f1": 0.0,
  "average_precision": 0.03333333333333333,
  "confusion_matrix_tn_fp_fn_tp": [
    [
      1160,
      0
    ],
    [
      40,
      0
    ]
  ]
}
logistic_regression {
  "threshold": 0.5,
  "precision": 0.14912280701754385,
  "recall": 0.85,
  "f1": 0.2537313432835821,
  "average_precision": 0.3742660671586222,
  "confusion_matrix_tn_fp_fn_tp": [
    [
      966,
      194
    ],
    [
      6,
      34
    ]
  ]
}
random_forest {
  "threshold": 0.803118076109186,
  "precision": 0.5897435897435898,
  "recall": 0.575,
  "f1": 0.5822784810126582,
  "average_precision": 0.65250377425591,
  "confusion_matrix_tn_fp_fn_tp": [
    [
      1144,
      16
    ],
    [
      17,
      23
    ]
  ]
}


## Ondorioak eta mugak

RFk 23/40 hutsegite hautematen ditu eta 17 galtzen ditu. Logistikoak 34/40 hautematen ditu baina 194 alarma faltsu sortzen ditu.
F1 bidezko hautaketa akademikoa ez da alerta-politika operatiboaren balioztatzea.
Ez dago timestampik, mantentze-historiarik edo 24 orduko jarraipenik. Ez dago kalibrazio egiaztaturik.
CG06ko Parquet/DuckDB eta CG09ko Isolation Forest egiteke; API integrazioa eta liburutegien parekotasuna ere bai.